In [ ]:
from collections.abc import Sequence

import polars as pl
from polars import DataFrame
import scanpy as sc
from lets_plot import *

import cellestial as cl
from lets_plot.plot.core import LayerSpec, PlotSpec
from cellestial import retrieve

LetsPlot.setup_html()

import numpy as np
from lets_plot import *
from scipy.stats import gaussian_kde
from skimage import measure

data = sc.read("data/pbmc3k_pped.h5ad")

In [ ]:
dim = cl.dimensional(
    data,
    dimensions="umap",
    key="cell_type_lvl1",
    size=1,
    axis_type="arrow",
    alpha=0.6,
    tooltips=["cell_type_lvl1", "n_genes", "total_counts_hb"],
    legend_ondata=True,
    ondata_size=12,
    # ondata_color="black",
    ondata_fontface="bold",
    ondata_family="mono",
    ondata_alpha=0.8,
)
dim += ggsize(800, 600)
dim

In [ ]:
frame = cl.retrieve(dim)

In [ ]:
def _get_density_boundary_skimage(
    frame: DataFrame,
    x: str,
    y: str,
    group_by: str,
    groups: str | Sequence[str | Sequence[str]],
    padding: float = 1,
    level: float = 0.1,
    grid_size: int = 200,
) -> pl.DataFrame:
    """Creates and Returns a DataFrame to encircle the cluster via `geom_path`."""
    if isinstance(groups, str):
        groups = [groups]

    boundaries = []
    for group in groups:
        # 1. Isolate cluster points
        if isinstance(group, str):
            _group = [group]
        elif isinstance(group, Sequence):
            _group = list(group)
        points = frame.filter(pl.col(group_by).is_in(_group)).select([x, y]).to_numpy()
        if len(points) < 5:
            continue

        # 2. KDE Calculation
        kde = gaussian_kde(points.T)

        # 3. Create Grid
        x_min, y_min = points.min(axis=0) - padding
        x_max, y_max = points.max(axis=0) + padding

        x_range = np.linspace(x_min, x_max, grid_size)
        y_range = np.linspace(y_min, y_max, grid_size)
        xi, yi = np.meshgrid(x_range, y_range)

        # 4. Evaluate Density
        zi = kde(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)

        # 5. Extract Contours (The skimage magic)
        threshold = zi.max() * level
        # find_contours returns a list of [row, col] arrays
        contours = measure.find_contours(zi, threshold)

        for i, contour in enumerate(contours):
            # Map grid indices back to DIM coordinates
            # Note: skimage returns (row, col) which maps to (y_index, x_index)
            actual_x = np.interp(contour[:, 1], np.arange(grid_size), x_range)
            actual_y = np.interp(contour[:, 0], np.arange(grid_size), y_range)

            boundaries.append(
                pl.DataFrame(
                    {
                        x: actual_x,
                        y: actual_y,
                        group_by: [group] * len(actual_x),
                        "path": [f"{group}_{i}"] * len(actual_x),
                    }
                )
            )

    return pl.concat(boundaries)

In [ ]:
frame.get_column("cell_type_lvl1").unique().to_list()

In [ ]:
#'Lymphocytes', 'Monocytes', 'Erythroid'

# --- Execution ---

density_frame = _get_density_boundary_skimage(
    frame,
    x="X_UMAP1",
    y="X_UMAP2",
    group_by="cell_type_lvl1",
    groups=["Lymphocytes", "Monocytes", "Erythroid"],
    padding=1,
    level=0.04,
    grid_size=300,
)

# Add the paths to your existing dim object
dim + geom_path(
    data=density_frame,
    mapping=aes(x="X_UMAP1", y="X_UMAP2", group="path"),
    color="black",
    linetype="dashed",
    size=1,
)
density_frame

In [ ]:
#'Lymphocytes', 'Monocytes', 'Erythroid'

# --- Execution ---

density_frame = _get_density_boundary_skimage(
    frame,
    x="X_UMAP1",
    y="X_UMAP2",
    group_by="cell_type_lvl1",
    groups=["Lymphocytes", "Monocytes", "Erythroid"],
    padding=1,
    level=0.04,
    grid_size=300,
)

# Add the paths to your existing dim object
dim + geom_path(
    data=density_frame,
    mapping=aes(x="X_UMAP1", y="X_UMAP2", group="path"),
    color="black",
    linetype="dashed",
    size=1,
)

In [ ]:
#'Lymphocytes', 'Monocytes', 'Erythroid'

# --- Execution ---

density_frame = _get_density_boundary_skimage(
    frame,
    x="X_UMAP1",
    y="X_UMAP2",
    group_by="cell_type_lvl1",
    groups=[["Lymphocytes", "Monocytes"], ["Erythroid"]],
    padding=1,
    level=0.04,
    grid_size=300,
)

# Add the paths to your existing dim object
dim + geom_path(
    data=density_frame,
    mapping=aes(
        x="X_UMAP1",
        y="X_UMAP2",
        group="path",
    ),
    color="black",
    linetype="dashed",
    size=1,
)

In [ ]:
cl.get_mapping(dim)

In [ ]:
def _encircle_clusters(
    frame: DataFrame,
    x: str,
    y: str,
    group_by: str,
    groups: str | Sequence[str | Sequence[str]],
    padding: float = 1,
    level: float = 0.1,
    grid_size: int = 200,
    color: str = "#1f1f1f",
    linetype: str = "dashed",
    size: float = 1,
    **geom_path_kwargs,
) -> LayerSpec:
    frame = _get_density_boundary_skimage(
        frame,
        x=x,
        y=y,
        group_by=group_by,
        groups=groups,
        padding=padding,
        level=level,
        grid_size=grid_size,
    )

    return geom_path(
        data=frame,
        mapping=aes(x=x, y=y, group="path"),
        color=color,
        linetype=linetype,
        size=size,
    )


In [ ]:
dim + _encircle_clusters(
    frame=frame,
    x="X_UMAP1",
    y="X_UMAP2",
    group_by="cell_type_lvl1",
    groups=["Lymphocytes", "Monocytes", "Erythroid"],
    padding=1,
    level=0.04,  # Match your manual value!
    grid_size=300,  # Match your manual value!
)

In [ ]:
density_frame.unique([ "path"])

In [ ]:
vars(vars(dim).get("_FeatureSpec__props").get("mapping"))

In [ ]:
dim.as_dict().get("mapping")

In [ ]:
{**dim.as_dict().get("mapping"),**dim.as_dict().get("layers")[0].get("mapping")}

In [ ]:
def get_mapping(plot: PlotSpec, *, index=1) -> dict:
    """Returns the mapping of the plot."""
    return {**plot.as_dict().get("mapping"),**plot.as_dict().get("layers")[0].get("mapping")}

In [ ]:
get_mapping(dim)

In [ ]:
def outline_clusters(
    plot: PlotSpec,
    /,
    groups: str | Sequence[str | Sequence[str]],
    *,
    padding: float = 1.5,
    level: float = 0.04,
    grid_size: int = 200,
    color: str = "#1f1f1f",
    linetype: str = "dashed",
    size: float = 1,
    group_by: str | None = None,
    x: str | None = None,
    y: str | None = None,
    **geom_kwargs,
) -> LayerSpec:
    """Returns a Layer that outlines the given clusters."""
    _mapping = get_mapping(plot)
    x = _mapping.get("x") if x is None else x
    y = _mapping.get("y") if y is None else y
    group_by = _mapping.get("color") if group_by is None else group_by

    frame = retrieve(plot)

    # get boundaries
    _frame = _get_density_boundary_skimage(
        frame,
        x=x,
        y=y,
        group_by=group_by,
        groups=groups,
        padding=padding,
        level=level,
        grid_size=grid_size,
    )

    return geom_path(
        data=_frame,
        mapping=aes(x=x, y=y, group="path"),
        color=color,
        linetype=linetype,
        size=size,
    )


In [ ]:
dim + cl.outline_clusters(dim,groups=["Lymphocytes", "Monocytes","Erythroid"])

In [ ]:
aes("X_UMAP1").as_dict()

In [ ]:
type(aes())

In [ ]:
dim.as_dict().get("data")